# Can we predict 30-day readmission?
Target: `readmit_30d`. Data: 11,868 hospital stays from 1,500 people (~8 stays each).

Plan: inspect data → check what structure could mislead us (repeat visits per person) → baseline → models evaluated with **person-grouped** cross-validation → judge whether the signal is real and useful.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',140)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data checks

In [ ]:
print(df.shape, 'missing:',df.isna().sum().sum(),'dupes:',df.duplicated().sum())
print('persons:',df.person_key.nunique()); print(df.groupby('person_key').size().describe())
print('Readmission rate: %.3f'%df.readmit_30d.mean())
df.describe().T

In [ ]:
# is the outcome a person-level trait? (would make random row splits optimistic)
pr=df.groupby('person_key').readmit_30d.agg(['mean','size'])
sim=np.random.default_rng(0).binomial(pr['size'],df.readmit_30d.mean())/pr['size']
print('SD of per-person rate: observed %.3f vs expected if no person effect %.3f'%(pr['mean'].std(),sim.std()))

## 2. Univariate look

In [ ]:
num=['birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b','visit_number','postcode_area']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,c in zip(ax.ravel(),num):
    q=pd.qcut(df[c],10,duplicates='drop'); gm=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(gm)),gm.values,'o-'); a.axhline(df.readmit_30d.mean(),color='gray',ls='--'); a.set_title(c); a.set_ylim(0,.4); a.set_xlabel('decile')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean())

## 3. Modelling setup
Rows from the same person are not independent, so CV is grouped by `person_key` (a random row split would put the same person in train and test). Metrics: ROC-AUC, average precision (base rate ≈ 0.20 is the no-skill level) and Brier score.
Postcode is treated as a categorical code, not a quantity.

In [ ]:
y=df.readmit_30d.values; g=df.person_key.values
cat=['sex','postcode_area']; numc=['birth_year','bmi','length_of_stay_days','n_medications','lab_a','lab_b','visit_number']
X=df[cat+numc].copy(); X['postcode_area']=X.postcode_area.astype(str)
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
pre=ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),('n',StandardScaler(),numc)])
pre_nopc=ColumnTransformer([('c',OneHotEncoder(),['sex']),('n',StandardScaler(),numc)])
models={'Base rate':make_pipeline(pre,DummyClassifier()),
 'Logistic (no postcode)':make_pipeline(pre_nopc,LogisticRegression(max_iter=1000)),
 'Logistic (all)':make_pipeline(pre,LogisticRegression(max_iter=1000,C=0.5)),
 'Gradient boosting':make_pipeline(ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat),('n','passthrough',numc)]),
        HistGradientBoostingClassifier(max_depth=3,learning_rate=0.03,max_iter=250,l2_regularization=5,random_state=0))}
oof={}; rows=[]
for n,m in models.items():
    p=cross_val_predict(m,X,y,cv=cv,groups=g,method='predict_proba')[:,1]; oof[n]=p
    f=[roc_auc_score(y[te],p[te]) for _,te in cv.split(X,y,g)]
    rows.append((n,roc_auc_score(y,p),np.std(f),average_precision_score(y,p),brier_score_loss(y,p)))
res=pd.DataFrame(rows,columns=['model','AUC','AUC fold SD','PR-AUC','Brier']).set_index('model'); res.round(3)

## 4. Is the signal real? Label-permutation test

In [ ]:
rng=np.random.default_rng(1); m=models['Logistic (no postcode)']; null=[]
for i in range(30):
    ys=rng.permutation(y)
    p=cross_val_predict(m,X,ys,cv=StratifiedGroupKFold(5,shuffle=True,random_state=i),groups=g,method='predict_proba')[:,1]
    null.append(roc_auc_score(ys,p))
obs=res.loc['Logistic (no postcode)','AUC']
print('null AUC mean %.3f, max %.3f (n=30); observed %.3f'%(np.mean(null),np.max(null),obs))

## 4b. Person history (the outcome clusters within people)
Per-person readmission rates vary much more than chance would give, so people differ in baseline risk. A realistic predictor is the person's **earlier** stays' outcomes (known when a new stay is admitted). Features below use only visits with a smaller `visit_number` (strictly prior), so there is no leakage of the current row's label. Folds are still grouped by person.

In [ ]:
d=df.sort_values(['person_key','visit_number']).copy()
grp=d.groupby('person_key').readmit_30d
d['prior_n']=grp.cumcount()
d['prior_readmits']=grp.cumsum()-d.readmit_30d
d['prior_rate']=np.where(d.prior_n>0,d.prior_readmits/d.prior_n.clip(lower=1),np.nan)
d=d.loc[df.index]  # restore original row order
Xh=X.copy(); Xh['prior_n']=d.prior_n; Xh['prior_readmits']=d.prior_readmits; Xh['prior_rate']=d.prior_rate
# the same patient-level features plus history; HGB handles NaN (first visit)
hgb_h=make_pipeline(ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cat),('n','passthrough',numc+['prior_n','prior_readmits','prior_rate'])]),
        HistGradientBoostingClassifier(max_depth=3,learning_rate=0.03,max_iter=250,l2_regularization=5,random_state=0))
ph=cross_val_predict(hgb_h,Xh,y,cv=cv,groups=g,method='predict_proba')[:,1]; oof['GB + prior history']=ph
res.loc['GB + prior history']=[roc_auc_score(y,ph),np.std([roc_auc_score(y[te],ph[te]) for _,te in cv.split(Xh,y,g)]),average_precision_score(y,ph),brier_score_loss(y,ph)]
print(res.round(3))
# does history help only where it exists?
m2=d.prior_n.values>0
print('AUC on visits with history: without %.3f, with %.3f'%(roc_auc_score(y[m2],oof['Gradient boosting'][m2]),roc_auc_score(y[m2],ph[m2])))

## 5. What drives it, and how well does it work?

In [ ]:
best=models['Logistic (no postcode)'].fit(X,y)
coef=pd.Series(best[-1].coef_[0],index=best[0].get_feature_names_out()).sort_values()
print(coef.round(3))
pi=permutation_importance(models['Gradient boosting'].fit(X,y),X,y,scoring='roc_auc',n_repeats=5,random_state=0)
print(pd.Series(pi.importances_mean,index=X.columns).sort_values(ascending=False).round(4))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4.5))
for n in list(oof)[1:]:
    fp,tp,_=roc_curve(y,oof[n]); ax[0].plot(fp,tp,label=f'{n} ({roc_auc_score(y,oof[n]):.3f})')
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend(); ax[0].set_title('ROC (grouped CV)')
for n in list(oof)[1:]:
    a,b=calibration_curve(y,oof[n],n_bins=8,strategy='quantile'); ax[1].plot(b,a,'o-',label=n)
ax[1].plot([0,.5],[0,.5],'k--'); ax[1].set_xlabel('predicted'); ax[1].set_ylabel('observed'); ax[1].set_title('Calibration'); ax[1].legend()
plt.tight_layout(); plt.show()
p=oof['Logistic (no postcode)']; o=np.argsort(-p)
for k in (.1,.2,.3):
    top=o[:int(k*len(p))]; print(f'top {k:.0%} flagged: readmit rate {y[top].mean():.3f} (lift {y[top].mean()/y.mean():.2f}x), captures {y[top].sum()/y.sum():.0%} of readmissions')

## Conclusions (numbers from a run with the fixed seeds above)
* **Yes, there is real but modest signal from admission/stay features.** Grouped-CV ROC-AUC ≈ 0.65 for a plain logistic regression (no-skill = 0.50; 30 label-shuffled runs gave 0.499 mean, 0.516 max). PR-AUC ≈ 0.32 vs 0.20 base rate. Flagging the top 20% of stays catches ~34% of readmissions (lift ≈ 1.7x). Useful for rough risk ranking, not for confident individual calls.
* **Main drivers:** longer stay, more medications at discharge and higher `lab_a` raise risk; BMI/age/postcode/`lab_b`/sex/visit number contribute little. The simple logistic model beat gradient boosting and the postcode-categorical variants (those add noise), so the relationship looks close to linear.
* **People differ in baseline risk** (per-person rate SD 0.27 vs 0.15 expected by chance). Adding strictly-prior-visit outcomes lifts AUC to ≈ 0.82 (≈ 0.84 on visits that have history). This only works for returning patients, relies on `visit_number` being chronological, and should be checked with the data owners (e.g. whether the earlier outcome is truly known at the time of prediction). A hidden persistent patient factor likely sits behind it.
* **Evaluation notes:** a random row split would be optimistic here because the same person appears many times, so all CV is grouped by person. Probabilities were not recalibrated; Brier improves only slightly over the base rate for the stay-feature models.
* **Not done:** time-based validation (no dates provided), threshold choice tied to intervention costs, subgroup fairness checks.